#### Prepare files

In [16]:
import json
import pandas as pd

records = []
pred_dir = "/home/csgrad/mbhosale/phd/MrFair/LLaVA-Rad/results/llavarad_fair_mi" 
with open(pred_dir+"/merged_pred.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)      # urlsafe json parser, keeps strings intact
        records.append(rec)

# Build a DataFrame out of it
df = pd.DataFrame.from_records(records)
# "id" (this id is patient ID and Study ID) We need patient ID only for fairness analysis
df['subject_id'] = df['id'].apply(lambda x: int(x.split("_")[0]))

In [17]:
df

,id,query,reference,prediction,subject_id
0,10046166_57379357,<image>\nProvide a description of the findings...,Frontal and lateral views of the chest were ob...,The patient is status post median sternotomy a...,10046166
1,10046166_57977208,<image>\nProvide a description of the findings...,"In comparison with the study of ___, there is ...","In comparison with the study of ___, there is ...",10046166
2,10268877_50042142,<image>\nProvide a description of the findings...,The ET tube is 3.5 cm above the carina. The N...,Single portable view of the chest. Endotrache...,10268877
3,10268877_50239281,<image>\nProvide a description of the findings...,Left PICC tip is seen terminating in the regio...,"In comparison with the study of ___, there is ...",10268877
4,10268877_51513702,<image>\nProvide a description of the findings...,Single AP portable view of the chest. No prio...,Single portable view of the chest. The lungs ...,10268877
...,...,...,...,...,...
2456,15131736_52449022,<image>\nProvide a description of the findings...,"Lung volumes are low, similar when compared to...",The heart is severely enlarged. There is pulm...,15131736
2457,19075045_58071016,<image>\nProvide a description of the findings...,Sternotomy with valve prosthesis. Endotrachea...,The endotracheal tube is in appropriate positi...,19075045
2458,19991135_51777681,<image>\nProvide a description of the findings...,PA and lateral radiographs of the chest were a...,Frontal and lateral views of the chest were ob...,19991135
2459,12185775_57910301,<image>\nProvide a description of the findings...,The ET and NG tubes have been removed. A right...,Compared to the prior study there is no signif...,12185775


In [18]:
# Lets load the demographic data
patients_data = pd.read_csv("/a2il/data/mbhosale/MrFair/physionet.org/mimc-cxr-jpeg/patients.csv")

In [19]:
patients_data

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13
...,...,...,...,...,...,...
364622,19999828,F,46,2147,2017 - 2019,NaN
364623,19999829,F,28,2186,2008 - 2010,NaN
364624,19999840,M,58,2164,2008 - 2010,2164-09-17
364625,19999914,F,49,2158,2017 - 2019,NaN


In [20]:
merged_df = df.merge(patients_data, on='subject_id', how='left', validate='m:1')

In [21]:
merged_df[merged_df['dod'].isna()]

,id,query,reference,prediction,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
30,10449297_52837403,<image>\nProvide a description of the findings...,AP upright and lateral views of the chest were...,The heart size is mildly enlarged. The aorta ...,10449297,NaN,NaN,NaN,NaN,NaN
31,10449297_54773340,<image>\nProvide a description of the findings...,Comparison is made to ___. In\n comparison to...,Frontal and lateral views of the chest were ob...,10449297,NaN,NaN,NaN,NaN,NaN
32,10523725_56078456,<image>\nProvide a description of the findings...,Frontal and lateral views of the chest. The l...,The patient is status post median sternotomy a...,10523725,M,76.0,2138.0,2011 - 2013,NaN
33,10715477_50563564,<image>\nProvide a description of the findings...,"In comparison with the study of ___, there is ...","In comparison with the study of ___, the monit...",10715477,NaN,NaN,NaN,NaN,NaN
34,10715477_51185902,<image>\nProvide a description of the findings...,Right internal jugular sheath ends at upper SV...,"In comparison with the study of ___, there is ...",10715477,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2447,15114531_54616688,<image>\nProvide a description of the findings...,There is a left PICC which terminates within t...,The lungs are clear. The cardiomediastinal si...,15114531,F,60.0,2154.0,2008 - 2010,NaN
2448,17270742_50989704,<image>\nProvide a description of the findings...,Dominant central cavitary lesions are similar ...,The lungs are hyperinflated with bilateral dif...,17270742,NaN,NaN,NaN,NaN,NaN
2450,12699874_57974904,<image>\nProvide a description of the findings...,Again seen is a large right hydropneumothorax ...,PA and lateral views of the chest were obtaine...,12699874,M,85.0,2129.0,2011 - 2013,NaN
2451,15881535_58897728,<image>\nProvide a description of the findings...,"The cardiomediastinal silhouette, pulmonary va...",The lungs are clear without focal consolidatio...,15881535,M,66.0,2184.0,2008 - 2010,NaN


In [22]:
merged_df['gender'].isna().value_counts()

gender
False    2332
True      129
Name: count, dtype: int64

In [23]:
merged_df['anchor_age'].isna().value_counts()

anchor_age
False    2332
True      129
Name: count, dtype: int64

In [24]:
merged_df['dod'].isna().value_counts()

dod
False    1506
True      955
Name: count, dtype: int64

In [25]:
# We do need age and gender for fairness analysis, so we will just drop the rows missing
merged_df = merged_df.dropna(subset=['anchor_age', 'gender'])

In [26]:
merged_df['gender'].value_counts()

gender
M    1301
F    1031
Name: count, dtype: int64

In [27]:
import numpy as np
bins   = [0, 44, 65, np.inf]
labels = ['0–44', '44–65', '65+']
merged_df = merged_df.copy()

merged_df.loc[:, 'age_group'] = pd.cut(
    merged_df['anchor_age'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)


In [28]:
merged_df['age_group'].value_counts()

age_group
65+      1151
44–65    1080
0–44      101
Name: count, dtype: int64

In [29]:
# Lets separate out the results according to demographic groups
merged_df[merged_df['gender'] == 'M'][['id','query','reference','prediction']] \
    .to_json(pred_dir+'/gender_m.jsonl', orient='records', lines=True)

merged_df[merged_df['gender'] == 'F'][['id','query','reference','prediction']] \
    .to_json(pred_dir+'/gender_f.jsonl', orient='records', lines=True)

merged_df[merged_df['age_group'] == '0–44'][['id','query','reference','prediction']] \
    .to_json(pred_dir+'/age_0_44.jsonl', orient='records', lines=True)

merged_df[merged_df['age_group'] == '44–65'][['id','query','reference','prediction']] \
    .to_json(pred_dir+'/age_44_65.jsonl', orient='records', lines=True)

merged_df[merged_df['age_group'] == '65+'][['id','query','reference','prediction']] \
    .to_json(pred_dir+'/age_65_inf.jsonl', orient='records', lines=True)


In [30]:
admissions = pd.read_csv("/a2il/data/mbhosale/MrFair/physionet.org/mimc-cxr-jpeg/admissions.csv")

In [31]:
admissions['insurance'].value_counts()

insurance
Medicare     244576
Private      173399
Medicaid     104229
Other         14006
No charge       463
Name: count, dtype: int64

In [32]:
admissions['race'].value_counts()

race
WHITE                                        336538
BLACK/AFRICAN AMERICAN                        75482
OTHER                                         19788
WHITE - OTHER EUROPEAN                        13972
UNKNOWN                                       13870
HISPANIC/LATINO - PUERTO RICAN                10903
HISPANIC OR LATINO                             8287
ASIAN                                          7809
ASIAN - CHINESE                                7644
WHITE - RUSSIAN                                6597
BLACK/CAPE VERDEAN                             6205
HISPANIC/LATINO - DOMINICAN                    6070
BLACK/CARIBBEAN ISLAND                         3875
BLACK/AFRICAN                                  3495
UNABLE TO OBTAIN                               3478
PATIENT DECLINED TO ANSWER                     2162
PORTUGUESE                                     2082
ASIAN - SOUTH EAST ASIAN                       1973
WHITE - EASTERN EUROPEAN                       1886
HISPANI

In [33]:
# We need to p
import numpy as np
race = admissions['race']

# define boolean masks
conds = [
    race.str.contains(r'^WHITE', case=False, na=False),
    race.str.contains(r'^BLACK', case=False, na=False),
    race.str.contains(r'^ASIAN', case=False, na=False),
    race.str.contains(r'HISPANIC', case=False, na=False),
    race.str.contains(r'AMERICAN INDIAN', case=False, na=False),
    race.str.contains(r'HAWAIIAN|PACIFIC ISLANDER', case=False, na=False),
    race.str.contains(r'^OTHER$', case=False, na=False),
    race.str.contains(r'DECLINED|UNABLE TO OBTAIN', case=False, na=False),
    race.str.contains(r'^UNKNOWN$', case=False, na=False),
]

# corresponding labels
labels = [
    'White',
    'Black or African American',
    'Asian',
    'Hispanic or Latino',
    'American Indian or Alaska Native',
    'Native Hawaiian or Pacific Islander',
    'Other',
    'Declined / Unable to obtain',
    'Unknown',
]
admissions['race_major'] = np.select(conds, labels, default='Other')
admissions['race_major'].value_counts()

race_major
White                                  360519
Black or African American               89057
Hispanic or Latino                      32210
Other                                   23240
Asian                                   19751
Unknown                                 13870
Declined / Unable to obtain              5640
American Indian or Alaska Native         1247
Native Hawaiian or Pacific Islander       494
Name: count, dtype: int64

In [34]:
admissions['marital_status'].value_counts()

marital_status
MARRIED     229134
SINGLE      206232
WIDOWED      56687
DIVORCED     40356
Name: count, dtype: int64

In [35]:
admissions = (admissions.drop_duplicates(subset=['subject_id'], keep='first')[['subject_id','insurance','race_major','marital_status']])

In [36]:
admissions['insurance'].value_counts()

insurance
Private      86139
Medicare     85524
Medicaid     38022
Other         6952
No charge      187
Name: count, dtype: int64

In [37]:
admissions['race_major'].value_counts()

race_major
White                                  147528
Black or African American               28659
Hispanic or Latino                      11981
Other                                   10766
Unknown                                 10352
Asian                                    9660
Declined / Unable to obtain              3771
American Indian or Alaska Native          481
Native Hawaiian or Pacific Islander       254
Name: count, dtype: int64

In [38]:
admissions['marital_status'].value_counts()

marital_status
MARRIED     95614
SINGLE      82797
WIDOWED     20226
DIVORCED    14119
Name: count, dtype: int64

In [39]:
merged_df = merged_df.merge(admissions, on='subject_id', how='left', validate='m:1')

In [40]:
merged_df['insurance'].value_counts()

insurance
Medicare     1552
Private       479
Medicaid      276
Other          21
No charge       4
Name: count, dtype: int64

In [41]:
merged_df['race_major'].value_counts()

race_major
White                               1621
Black or African American            489
Asian                                 85
Hispanic or Latino                    67
Other                                 37
Unknown                               22
Declined / Unable to obtain            7
American Indian or Alaska Native       4
Name: count, dtype: int64

In [42]:
merged_df['marital_status'].value_counts()

marital_status
MARRIED     1046
SINGLE       703
WIDOWED      344
DIVORCED     239
Name: count, dtype: int64

In [43]:
merged_df[merged_df['race_major'] == 'White'][['id','query','reference','prediction']].to_json(pred_dir+'/race_white.jsonl', orient='records', lines=True)
merged_df[merged_df['race_major'] == 'Black or African American'][['id','query','reference','prediction']].to_json(pred_dir+'/race_black.jsonl', orient='records', lines=True)
merged_df[merged_df['race_major'] == 'Asian'][['id','query','reference','prediction']].to_json(pred_dir+'/race_asian.jsonl', orient='records', lines=True)
merged_df[merged_df['race_major'] == 'Other'][['id','query','reference','prediction']].to_json(pred_dir+'/race_other.jsonl', orient='records', lines=True)
merged_df[merged_df['race_major'] == 'Hispanic or Latino'][['id','query','reference','prediction']].to_json(pred_dir+'/race_hispanic.jsonl', orient='records', lines=True)


In [44]:
merged_df.to_json('../results/llavarad/merged_demographics.jsonl', orient='records', lines=True)

### Fairness Analysis

#### 1. Gender

In [83]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/gender_m.jsonl --run_name gender_m

[2025-07-03 20:59:47,344] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  98%|████████████▊| 491/500 [00:04<00:00, 115.43it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:04<00:00, 114.73it/s]


bootstrap macro f1 95% CI:   2%|▎             | 11/500 [00:00<00:04, 105.75it/s]

bootstrap macro f1 95% CI:   5%|▋             | 23/500 [00:00<00:04, 112.37it/s]

bootstrap macro f1 95% CI:   7%|▉             | 35/500 [00:00<00:04, 112.77it/s]

bootstrap macro f1 95% CI:   9%|█▎            | 47/500 [00:00<00:03, 1

In [84]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/gender_f.jsonl --run_name gender_f

[2025-07-03 21:03:20,671] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  99%|████████████▊| 493/500 [00:03<00:00, 141.29it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:03<00:00, 140.51it/s]


bootstrap macro f1 95% CI:   3%|▎             | 13/500 [00:00<00:03, 127.26it/s]

bootstrap macro f1 95% CI:   6%|▊             | 28/500 [00:00<00:03, 136.74it/s]

bootstrap macro f1 95% CI:   9%|█▏            | 43/500 [00:00<00:03, 139.39it/s]

bootstrap macro f1 95% CI:  12%|█▌            | 58/500 [00:00<00:03, 1

#### 2. Age Groups

In [85]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/age_0_44.jsonl --run_name age_0_44

[2025-07-03 21:10:06,605] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  87%|███████████▎ | 434/500 [00:00<00:00, 721.24it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:00<00:00, 717.86it/s]
/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [86]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/age_44_65.jsonl --run_name age_44_65

[2025-07-03 21:10:47,720] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  98%|████████████▋| 489/500 [00:03<00:00, 136.13it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:03<00:00, 136.16it/s]


bootstrap macro f1 95% CI:   3%|▎             | 13/500 [00:00<00:03, 124.45it/s]

bootstrap macro f1 95% CI:   5%|▊             | 27/500 [00:00<00:03, 131.76it/s]

bootstrap macro f1 95% CI:   8%|█▏            | 41/500 [00:00<00:03, 134.21it/s]

bootstrap macro f1 95% CI:  11%|█▌            | 55/500 [00:00<00:03, 1

In [87]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/age_65_inf.jsonl --run_name age_65_inf

[2025-07-03 21:12:20,971] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  98%|████████████▊| 491/500 [00:03<00:00, 130.29it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:03<00:00, 129.34it/s]


bootstrap macro f1 95% CI:   2%|▎             | 12/500 [00:00<00:04, 116.59it/s]

bootstrap macro f1 95% CI:   5%|▋             | 25/500 [00:00<00:03, 123.96it/s]

bootstrap macro f1 95% CI:   8%|█             | 38/500 [00:00<00:03, 126.45it/s]

bootstrap macro f1 95% CI:  10%|█▍            | 52/500 [00:00<00:03, 1

#### 3. Race

In [47]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/race_white.jsonl --run_name race_white

[2025-07-04 18:23:21,280] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI: 100%|█████████████▉| 499/500 [00:05<00:00, 93.62it/s]

bootstrap micro f1 95% CI: 100%|██████████████| 500/500 [00:05<00:00, 93.73it/s]


bootstrap macro f1 95% CI:   2%|▎               | 9/500 [00:00<00:05, 85.59it/s]

bootstrap macro f1 95% CI:   4%|▌              | 19/500 [00:00<00:05, 91.40it/s]

bootstrap macro f1 95% CI:   6%|▊              | 29/500 [00:00<00:05, 93.34it/s]

bootstrap macro f1 95% CI:   8%|█▏             | 39/500 [00:00<00:04, 

In [48]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/race_black.jsonl --run_name race_black

[2025-07-04 18:26:37,112] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  97%|████████████▌| 485/500 [00:01<00:00, 262.68it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:01<00:00, 262.23it/s]


bootstrap macro f1 95% CI:   5%|▊             | 27/500 [00:00<00:01, 261.78it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. 

In [49]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/race_asian.jsonl --run_name race_asian

[2025-07-04 18:27:30,032] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  93%|████████████ | 465/500 [00:00<00:00, 770.90it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:00<00:00, 767.48it/s]
/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [50]:
!python ../llava/eval/rrg_eval/run.py ../results/llavarad/race_hispanic.jsonl --run_name race_hispanic

[2025-07-04 18:27:55,769] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
CheXbert:   0%|                                           | 0/5 [00:00<?, ?it/s]/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

bootstrap micro f1 95% CI:  87%|███████████▎ | 435/500 [00:00<00:00, 871.06it/s]

bootstrap micro f1 95% CI: 100%|█████████████| 500/500 [00:00<00:00, 863.75it/s]
/home/csgrad/mbhosale/anaconda3/envs/llavarad/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [ ]:
# Lets check the 